<a href="https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding 1: ranking position is strongly related to CTR.** The paper reports a negative position/CTR correlation of about -0.46 and says expected CTR is the strongest feature in its LightGBM model. This is a useful descriptive finding: search position and clicks move together in the observed sample. My constructive methodology question is: **where does the label come from, and is it separated from the feature window?** The paper predicts CTR at page-date level, so I would document whether each target-day CTR uses only impressions and clicks observed on that day, while every rolling feature uses dates strictly before it. I would also report a volume floor because a tiny impression count can make CTR look extreme. The finding supports directional decision-support about which pages merit review; it does not by itself show that changing position or refreshing content causes more clicks.

**Finding 2: LightGBM has lower MAE than the organic click-curve baseline.** On the stated March 2026 out-of-time test, the paper reports MAE 0.00690 for LightGBM versus 0.00748 for the curve, a measured reduction of about 7.8%, while the curve has slightly higher R-squared. My constructive methodology question is: **does the validation design carry the claim for new pages and future periods?** The chronological split is directionally appropriate, but I would add a client-group check or report results by client and by impression volume. I would also confirm that the click curve and all target-derived quantities were fitted on training data only, and that the 49 recommendations were generated only from sealed test predictions. That would distinguish a measured out-of-time improvement in this snapshot from a broader claim about future traffic recovery or the causal value of refreshing content.

These questions are about scope and reproducibility, not about grading the paper. They are the same questions I apply to my own Week-5 result below.

In [10]:
# Code
from pathlib import Path
import numpy as np
import pandas as pd

CSV_URL = "https://raw.githubusercontent.com/SubhadeepBhadra/subhflyrank-internship/main/data/raw/content_refresh_anonymized.csv"
local_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
]
source_path = next((path for path in local_candidates if path.exists()), None)
df = pd.read_csv(source_path if source_path else CSV_URL)

required = {"content_id", "client_id", "impressions_90d", "content_age_days", "trend_direction"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Keep the same starter population rule as Week 5, but treat the label as a proxy.
frame = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").copy()
frame["is_declining_label"] = frame["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print(f"Rows: {len(frame):,} | clients: {frame['client_id'].nunique():,} | proxy decline rate: {frame['is_declining_label'].mean():.3f}")
print("Paper finding check: the paper reports position/CTR correlation about -0.46 and a 7.8% MAE reduction on its March 2026 test.")

Rows: 30,000 | clients: 32 | proxy decline rate: 0.542
Paper finding check: the paper reports position/CTR correlation about -0.46 and a 7.8% MAE reduction on its March 2026 test.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [11]:
# Code
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# These are observable before a future decision. Trend fields and the two comparison
# windows are excluded because they define the starter proxy label.
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
numeric_features = [column for column in numeric_features if column in frame.columns]
categorical_features = [column for column in categorical_features if column in frame.columns]

X_numeric = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
X_categorical = frame[categorical_features].fillna("unknown").astype(str)
X = pd.concat(
    [X_numeric.reset_index(drop=True), pd.get_dummies(X_categorical, prefix=categorical_features, dtype=float).reset_index(drop=True)],
    axis=1,
)
y = frame["is_declining_label"].to_numpy()
groups = frame["client_id"].fillna("unknown").astype(str).to_numpy()

# A transparent rule baseline: visible, stale, and relatively near the top.
visibility = np.log1p(frame["impressions_90d"].astype(float))
visibility = visibility / visibility.max()
freshness = frame["days_since_last_update"].clip(lower=0).astype(float)
freshness = freshness / max(freshness.max(), 1)
position = 1 / frame["avg_position"].clip(lower=1, upper=50).astype(float)
baseline_score = (visibility * (0.6 * freshness + 0.4 * position)).to_numpy()

def precision_at_50(labels, scores):
    top = np.argsort(-scores)[: min(50, len(scores))]
    return float(labels[top].mean())

def evaluate(labels, probabilities, baseline_scores):
    return {
        "roc_auc": roc_auc_score(labels, probabilities),
        "average_precision": average_precision_score(labels, probabilities),
        "precision_at_50": precision_at_50(labels, probabilities),
        "baseline_precision_at_50": precision_at_50(labels, baseline_scores),
        "base_rate": float(labels.mean()),
    }

def fit_and_score(train_idx, test_idx):
    model = RandomForestClassifier(
        n_estimators=160, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1,
    )
    model.fit(X.iloc[train_idx], y[train_idx])
    probabilities = model.predict_proba(X.iloc[test_idx])[:, 1]
    metrics = evaluate(y[test_idx], probabilities, baseline_score[test_idx])
    return model, probabilities, metrics

row_train, row_test = train_test_split(np.arange(len(frame)), test_size=0.2, random_state=42, stratify=y)
group_train, group_test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups=groups))
row_model, row_probabilities, row_metrics = fit_and_score(row_train, row_test)
group_model, group_probabilities, group_metrics = fit_and_score(group_train, group_test)

comparison = pd.DataFrame([
    {"split": "Week-5-style random row holdout", **row_metrics},
    {"split": "honest client-grouped holdout", **group_metrics},
]).set_index("split")
print(comparison.round(3).to_string())
print(f"\nGrouped test clients: {len(set(groups[group_test]))}; overlap with train: {len(set(groups[group_train]) & set(groups[group_test]))}")

# Show real mistakes without printing IDs, URLs, titles, client names, or queries.
group_predictions = (group_probabilities >= 0.5).astype(int)
error_mask = group_predictions != y[group_test]
error_examples = frame.iloc[group_test[error_mask]][
    ["impressions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update", "trend_direction"]
].head(3).copy()
error_examples.insert(0, "masked_test_row", np.arange(1, len(error_examples) + 1))
print("\nThree grouped-test errors, with identifiers masked:")
print(error_examples.to_string(index=False))

# The grouped result is the headline result because client-level repetition is plausible.
audit_results = {"row": row_metrics, "grouped": group_metrics, "feature_columns": list(X.columns)}

                                 roc_auc  average_precision  precision_at_50  baseline_precision_at_50  base_rate
split                                                                                                            
Week-5-style random row holdout    0.759              0.770             0.88                      0.52      0.542
honest client-grouped holdout      0.609              0.591             0.56                      0.54      0.511

Grouped test clients: 7; overlap with train: 0

Three grouped-test errors, with identifiers masked:
 masked_test_row  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update trend_direction
               1            15320          20.3 0.05               445                      25            down
               2              307          39.8 0.00               238                     103          stable
               3              297          13.9 0.34               502                      20            down

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [12]:
# Code
# Leakage audit: explicit exclusions plus a deliberate attack.
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
comparison_windows = {
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}
product_outputs = {"health_score", "priority_score", "action_type", "refresh_tier"}
forbidden = label_derived | comparison_windows | product_outputs | {"content_id", "client_id"}
violations = sorted(forbidden.intersection(audit_results["feature_columns"]))
print("Final feature leakage check:", "PASS" if not violations else f"FAIL ({violations})")

# Add the known label source back on purpose. A large jump is the expected confession.
leaky_values = pd.to_numeric(frame["trend_pct"], errors="coerce").fillna(0).to_numpy()
leaky_model = RandomForestClassifier(
    n_estimators=80, max_depth=5, min_samples_leaf=25,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
leaky_model.fit(leaky_values[group_train].reshape(-1, 1), y[group_train])
leaky_probabilities = leaky_model.predict_proba(leaky_values[group_test].reshape(-1, 1))[:, 1]
leaky_auc = roc_auc_score(y[group_test], leaky_probabilities)
print(f"Honest grouped ROC AUC: {group_metrics['roc_auc']:.3f}")
print(f"Deliberately leaky grouped ROC AUC: {leaky_auc:.3f}")
print("Decision: keep the honest grouped score; trend_pct is derived from the same comparison used to make the label.")

assert not violations
assert "trend_direction" not in audit_results["feature_columns"]
assert "trend_pct" not in audit_results["feature_columns"]

Final feature leakage check: PASS
Honest grouped ROC AUC: 0.609
Deliberately leaky grouped ROC AUC: 1.000
Decision: keep the honest grouped score; trend_pct is derived from the same comparison used to make the label.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [13]:
# Code
bold_claim = "The random-forest model predicts which pages will recover traffic after a refresh and is ready for production scheduling."
careful_claim = (
    "On this anonymized starter snapshot, the model measured a ranked association with the proxy label "
    "and produced a client-grouped review score for decision-support. The result is directional and "
    "observational: it does not measure whether a refresh causes traffic recovery, and it should be "
    "checked by an editor against future data before operational use."
)
print("Original claim:")
print(bold_claim)
print("\nSafe rewrite:")
print(careful_claim)
print("\nRequired language present:", all(word in careful_claim for word in ["measured", "directional", "observational", "decision-support"]))
assert all(word in careful_claim for word in ["measured", "directional", "observational", "decision-support"])
assert "production" not in careful_claim.lower()

Original claim:
The random-forest model predicts which pages will recover traffic after a refresh and is ready for production scheduling.

Safe rewrite:
On this anonymized starter snapshot, the model measured a ranked association with the proxy label and produced a client-grouped review score for decision-support. The result is directional and observational: it does not measure whether a refresh causes traffic recovery, and it should be checked by an editor against future data before operational use.

Required language present: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.